In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Dataset, DataLoader
import copy
from tqdm.notebook import tqdm  # Specialized for Colab/Jupyter

# 1. Setup the GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Define GTSRB classes to keep
target_gtsrb_classes = [0, 1, 2, 3, 4, 5, 7, 8, 18, 19, 20, 23, 25, 26, 27, 30, 31, 33, 34]
num_classes = len(target_gtsrb_classes)
class_mapping = {old_id: new_id for new_id, old_id in enumerate(target_gtsrb_classes)}

# 3. The Custom Filter Wrapper
class FilteredGTSRB(Dataset):
    def __init__(self, original_dataset, mapping):
        self.dataset = original_dataset
        self.mapping = mapping
        print("Filtering images...")
        self.valid_indices = [
            i for i, (path, label) in enumerate(self.dataset._samples)
            if label in self.mapping
        ]

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        image, old_label = self.dataset[real_idx]
        new_label = self.mapping[old_label]
        return image, new_label

# 4. Data Augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 5. Load MobileNetV3-Small & Apply Fine-Tuning
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

# Freeze early layers, unfreeze late-stage features (Blocks 9-12)
for idx, child in enumerate(model.features.children()):
    if idx < 9:
        for param in child.parameters():
            param.requires_grad = False
    else:
        for param in child.parameters():
            param.requires_grad = True

# Adjust the final classifier head
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, num_classes)
model = model.to(device)

# 6. Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0005)

# 7. Load and Filter Datasets
print("Downloading and Preparing GTSRB...")
raw_train = datasets.GTSRB(root='./data', split='train', download=True, transform=train_transform)
raw_val = datasets.GTSRB(root='./data', split='test', download=True, transform=test_transform)

filtered_train = FilteredGTSRB(raw_train, class_mapping)
filtered_val = FilteredGTSRB(raw_val, class_mapping)

# SETTING num_workers=0 TO AVOID MULTIPROCESSING ERRORS
dataloaders = {
    'train': DataLoader(filtered_train, batch_size=32, shuffle=True, num_workers=0),
    'val': DataLoader(filtered_val, batch_size=32, shuffle=False, num_workers=0)
}

dataset_sizes = {'train': len(filtered_train), 'val': len(filtered_val)}

# 8. Training Loop
num_epochs = 50 # Reduced for example; keep 50 if you have time
best_acc = 0.0

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        # tqdm progress bar (leave=False keeps the output clean)
        pbar = tqdm(dataloaders[phase], desc=f"{phase.capitalize()}", leave=False)

        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f'{phase.capitalize()} Phase | Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            torch.save(model.state_dict(), 'best_filtered_mobilenet.pth')
            print(" >> New Best Model Saved << ")

print(f'Training complete! Best Val Acc: {best_acc:.4f}')

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Ensure the device is set up as in training
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Define the number of classes, matching your training setup
target_gtsrb_classes = [0, 1, 2, 3, 4, 5, 7, 8, 18, 19, 20, 23, 25, 26, 27, 30, 31, 33, 34]
num_classes = len(target_gtsrb_classes)

print("Loading the quantized MobileNetV3-Small architecture...")
# 1. Recreate the model architecture that was quantized
# Start with the original unquantized model first
model_eval = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

# Adjust the final classifier head to match training
in_features = model_eval.classifier[3].in_features
model_eval.classifier[3] = nn.Linear(in_features, num_classes)

# Apply dynamic quantization to this new model architecture
# This creates a model structure capable of loading quantized weights
quantized_model_eval = torch.quantization.quantize_dynamic(
    model_eval,
    {nn.Linear},
    dtype=torch.qint8
)

# Load the saved state_dict into the quantized model
quantized_model_eval.load_state_dict(torch.load('quantized_mobilenet.pth', map_location='cpu')) # Load to CPU
quantized_model_eval.eval()
quantized_model_eval = quantized_model_eval.to('cpu') # Ensure model is on CPU for evaluation

print("Evaluating quantized model accuracy...")

running_corrects = 0

# Iterate over the validation data
# We use the existing 'dataloaders' and 'dataset_sizes' from the training script
for inputs, labels in dataloaders['val']:
    inputs, labels = inputs.to('cpu'), labels.to('cpu') # Move inputs and labels to CPU

    with torch.no_grad(): # No need to calculate gradients for evaluation
        outputs = quantized_model_eval(inputs)
        _, preds = torch.max(outputs, 1)

    running_corrects += torch.sum(preds == labels.data)

accuracy = running_corrects.double() / dataset_sizes['val']

print(f'Quantized Model Accuracy on Validation Set: {accuracy:.4f}')

In [ ]:
from google.colab import files

# Download the quantized model
files.download('quantized_mobilenet.pth')
print("Quantized model download triggered! Check your browser's download folder.")